# Семинар 7. Обучение без учителя

В этом семинаре мы разберем:
- Кластеризация (KMeans, DBSCAN, иерархическая кластеризация)
- Снижение размерности (PCA, t-SNE, UMAP)
- Обнаружение аномалий (LOF)

In [ ]:
# Colab: install deps; locally use `uv run jupyter lab seminar.ipynb`
import sys
if "google.colab" in sys.modules:
    !pip install -q umap-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import (
    make_blobs, make_moons, make_circles, load_iris, load_digits,
)
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import LocalOutlierFactor
from scipy.cluster.hierarchy import dendrogram, linkage
import umap

np.random.seed(42)

## 1. Кластеризация

### 1.1 KMeans

Алгоритм:
1. Случайно инициализируем K центроидов
2. Присваиваем каждую точку ближайшему центроиду
3. Пересчитываем центроиды как среднее точек кластера
4. Повторяем 2-3 до сходимости

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=42)

km = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = km.fit_predict(X_blobs)

plt.figure(figsize=(10, 7))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='tab10', s=20, alpha=0.7)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='red', marker='X', s=200, edgecolors='k', label='centroids')
plt.title(f'KMeans (K=4, silhouette={silhouette_score(X_blobs, labels):.3f})')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

#### Elbow method и Silhouette score

Как выбрать K? Два подхода:
- **Elbow method**: ищем "локоть" на графике inertia (сумма квадратов расстояний до центроидов)
- **Silhouette score**: мера того, насколько точки похожи на свой кластер по сравнению с соседними

In [ ]:
K_range = range(2, 10)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blobs)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'o-')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow method')
axes[0].grid(True)

axes[1].plot(K_range, silhouettes, 'o-')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette analysis')
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 1.2 DBSCAN

Density-based clustering: не нужно указывать K. Находит области высокой плотности, разделенные областями низкой плотности. Умеет находить кластеры произвольной формы и отмечать выбросы.

Параметры:
- `eps` - радиус окрестности
- `min_samples` - минимум точек в окрестности для core point

In [ ]:
X_moon, y_moon = make_moons(n_samples=500, noise=0.1, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# KMeans fails on moons
km_labels = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_moon)
axes[0].scatter(X_moon[:, 0], X_moon[:, 1], c=km_labels, cmap='tab10', s=20)
axes[0].set_title('KMeans (K=2) - fails on non-convex')
axes[0].grid(alpha=0.2)

# DBSCAN works
db_labels = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_moon)
axes[1].scatter(X_moon[:, 0], X_moon[:, 1], c=db_labels, cmap='tab10', s=20)
axes[1].set_title(f'DBSCAN (eps=0.2) - handles moons')
axes[1].grid(alpha=0.2)

# DBSCAN with wrong eps
db_labels2 = DBSCAN(eps=0.5, min_samples=5).fit_predict(X_moon)
axes[2].scatter(X_moon[:, 0], X_moon[:, 1], c=db_labels2, cmap='tab10', s=20)
axes[2].set_title(f'DBSCAN (eps=0.5) - too large eps')
axes[2].grid(alpha=0.2)

plt.suptitle('KMeans vs DBSCAN on make_moons', fontsize=14)
plt.tight_layout()
plt.show()

### 1.3 Иерархическая кластеризация

Agglomerative (bottom-up): начинаем с N кластеров (каждая точка - кластер), на каждом шаге объединяем два ближайших. Результат визуализируется как дендрограмма.

In [ ]:
X_hier = X_blobs[:100]  # subset for readable dendrogram

Z = linkage(X_hier, method='ward')

plt.figure(figsize=(16, 6))
dendrogram(Z, truncate_mode='lastp', p=20)
plt.title('Dendrogram (Ward linkage, top 20 merges)')
plt.xlabel('Cluster')
plt.ylabel('Distance')
plt.grid(True, axis='y')
plt.show()

In [ ]:
# Разные методы связи (linkage)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, method in zip(axes, ['ward', 'complete', 'average']):
    agg = AgglomerativeClustering(n_clusters=4, linkage=method)
    labels = agg.fit_predict(X_blobs)
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='tab10', s=20)
    ax.set_title(f'Agglomerative ({method})')
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 1.4 Сравнение методов кластеризации

In [ ]:
# 4 разных типа данных
datasets = [
    ('Blobs', *make_blobs(n_samples=500, centers=3, cluster_std=1.0, random_state=42)),
    ('Moons', *make_moons(n_samples=500, noise=0.1, random_state=42)),
    ('Circles', *make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=42)),
    ('Anisotropic', *(lambda: (
        np.dot(make_blobs(n_samples=500, centers=3, cluster_std=0.8, random_state=42)[0],
               [[0.6, -0.6], [-0.4, 0.8]]),
        make_blobs(n_samples=500, centers=3, cluster_std=0.8, random_state=42)[1]
    ))()),
]

clusterers = [
    ('KMeans', lambda X: KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X)),
    ('DBSCAN', lambda X: DBSCAN(eps=0.5, min_samples=5).fit_predict(X)),
    ('Agglomerative', lambda X: AgglomerativeClustering(n_clusters=3).fit_predict(X)),
]

fig, axes = plt.subplots(len(datasets), len(clusterers), figsize=(18, 20))

for i, (dname, X, y_true) in enumerate(datasets):
    X_scaled = StandardScaler().fit_transform(X)
    for j, (cname, cluster_fn) in enumerate(clusterers):
        labels = cluster_fn(X_scaled)
        axes[i][j].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, cmap='tab10', s=10)
        axes[i][j].set_title(f'{cname}' if i == 0 else '')
        if j == 0:
            axes[i][j].set_ylabel(dname, fontsize=14)
        axes[i][j].grid(alpha=0.2)

plt.suptitle('Comparison of clustering methods', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 2. Снижение размерности

### 2.1 PCA (Principal Component Analysis)

Находит направления максимальной дисперсии (главные компоненты) и проецирует данные на них. Линейный метод, быстрый, интерпретируемый.

In [ ]:
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
y_iris = iris.target

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for cls in np.unique(y_iris):
    mask = y_iris == cls
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=iris.target_names[cls], s=30)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[0].set_title('PCA: Iris (4D -> 2D)')
axes[0].legend()
axes[0].grid(alpha=0.2)

# Scree plot
pca_full = PCA().fit(X_iris)
axes[1].bar(range(1, 5), pca_full.explained_variance_ratio_, alpha=0.7, label='Individual')
axes[1].plot(range(1, 5), np.cumsum(pca_full.explained_variance_ratio_), 'ro-', label='Cumulative')
axes[1].set_xlabel('Component')
axes[1].set_ylabel('Explained variance ratio')
axes[1].set_title('Scree plot')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 2.2 t-SNE

Нелинейный метод, сохраняет локальную структуру: близкие точки остаются близкими. Хорош для визуализации высокоразмерных данных.

Параметр `perplexity` ~ количество ближайших соседей, которые учитываются.

In [ ]:
digits = load_digits()
X_digits = StandardScaler().fit_transform(digits.data)
y_digits = digits.target
print(f'Digits: {X_digits.shape[0]} samples, {X_digits.shape[1]} features, {len(np.unique(y_digits))} classes')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, perp in zip(axes, [5, 30, 100]):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42)
    X_tsne = tsne.fit_transform(X_digits)
    scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
    ax.set_title(f't-SNE (perplexity={perp})')
    ax.grid(alpha=0.2)

plt.colorbar(scatter, ax=axes, shrink=0.6, label='Digit')
plt.suptitle('t-SNE on digits dataset (64D -> 2D)', fontsize=14)
plt.tight_layout()
plt.show()

**Внимание:** расстояния между кластерами в t-SNE НЕ информативны. Только расстояния внутри кластеров имеют смысл.

### 2.3 UMAP

Быстрая альтернатива t-SNE, лучше сохраняет глобальную структуру. Ключевые параметры:
- `n_neighbors` - размер локальной окрестности (~perplexity в t-SNE)
- `min_dist` - минимальное расстояние между точками в проекции

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, (n_neighbors, min_dist) in zip(axes, [(5, 0.1), (15, 0.1), (50, 0.5)]):
    reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, random_state=42)
    X_umap = reducer.fit_transform(X_digits)
    scatter = ax.scatter(X_umap[:, 0], X_umap[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
    ax.set_title(f'UMAP (n_neighbors={n_neighbors}, min_dist={min_dist})')
    ax.grid(alpha=0.2)

plt.colorbar(scatter, ax=axes, shrink=0.6, label='Digit')
plt.suptitle('UMAP on digits dataset (64D -> 2D)', fontsize=14)
plt.tight_layout()
plt.show()

### 2.4 Сравнение: PCA vs t-SNE vs UMAP

In [ ]:
X_pca_d = PCA(n_components=2).fit_transform(X_digits)
X_tsne_d = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_digits)
X_umap_d = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_digits)

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
for ax, X_proj, title in zip(axes,
    [X_pca_d, X_tsne_d, X_umap_d],
    ['PCA', 't-SNE', 'UMAP'],
):
    scatter = ax.scatter(X_proj[:, 0], X_proj[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
    ax.set_title(title, fontsize=14)
    ax.grid(alpha=0.2)

plt.colorbar(scatter, ax=axes, shrink=0.6, label='Digit')
plt.suptitle('Dimensionality reduction: digits (64D -> 2D)', fontsize=14)
plt.tight_layout()
plt.show()

| Метод | Линейный? | Скорость | Глобальная структура | Локальная структура | Использование |
|---|---|---|---|---|---|
| PCA | Да | Быстрый | Сохраняет | Средне | Preprocessing, шумоподавление |
| t-SNE | Нет | Медленный | Не сохраняет | Отлично | Визуализация |
| UMAP | Нет | Быстрый | Сохраняет лучше t-SNE | Отлично | Визуализация + downstream задачи |

## 3. Обнаружение аномалий

### Local Outlier Factor (LOF)

LOF оценивает "локальную плотность" каждой точки по сравнению с ее соседями. Точки с существенно меньшей плотностью - аномалии. В отличие от Isolation Forest (sem4), LOF учитывает локальный контекст.

In [ ]:
# Данные с выбросами
X_normal, _ = make_blobs(n_samples=300, centers=2, cluster_std=0.5, random_state=42)
rng = np.random.RandomState(42)
X_outliers = rng.uniform(low=-5, high=8, size=(20, 2))
X_lof = np.vstack([X_normal, X_outliers])

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.06)
lof_labels = lof.fit_predict(X_lof)
lof_scores = -lof.negative_outlier_factor_

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Predictions
axes[0].scatter(X_lof[:, 0], X_lof[:, 1], c=lof_labels, cmap='coolwarm', s=30)
axes[0].set_title('LOF predictions (red=outlier)')
axes[0].grid(alpha=0.2)

# Anomaly scores
scatter = axes[1].scatter(X_lof[:, 0], X_lof[:, 1], c=lof_scores, cmap='Reds', s=30)
plt.colorbar(scatter, ax=axes[1], label='LOF score')
axes[1].set_title('LOF anomaly scores (higher = more anomalous)')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

n_outliers_found = (lof_labels == -1).sum()
print(f'Найдено аномалий: {n_outliers_found} (добавлено: {len(X_outliers)})')

## Итоги

### Кластеризация
- **KMeans**: быстрый, но только выпуклые кластеры, нужно задавать K
- **DBSCAN**: произвольная форма кластеров, не нужно задавать K, чувствителен к eps
- **Hierarchical**: дает дендрограмму, можно выбрать K постфактум

### Снижение размерности
- **PCA**: быстрый, линейный, для preprocessing и визуализации
- **t-SNE**: нелинейный, только для визуализации, медленный
- **UMAP**: нелинейный, быстрее t-SNE, можно использовать для downstream задач

### Аномалии
- **Isolation Forest** (sem4): глобальная аномальность
- **LOF**: локальная аномальность (учитывает плотность соседей)